# 01 — Synthetic Data Generation & Exploration

Generate synthetic binary-classification datasets with **controlled distribution shifts**,
formatted identically to the real-arm data so the same demo-selection and prompting
pipeline can run unchanged.

**Shift types produced:**

- `synth_covariate` — P(X) changes, P(Y|X) preserved
- `synth_concept` — P(Y|X) changes (mechanism shift)
- `synth_spurious` — spurious feature correlation reverses


In [ ]:
import sys, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))

from src.data.generator import SyntheticTask, _sample_tasks
from src.data.synthetic_bridge import (
    FEATURE_NAMES, LABEL_TOKENS, SYNTHETIC_TASK_DESC,
    task_to_dataframes, save_dataset, make_codebook,
)

sns.set_theme(style="whitegrid", font_scale=1.1)
DATA_ROOT = Path.cwd().parent / "data" / "synthetic"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1. Generate tasks

We create **5 tasks per shift type** (different rule families and parameters)
so results aren't artifacts of one specific classification rule.
Each task produces a train pool (256 rows), ID test (100 rows), and OOD test (100 rows).


In [ ]:
from types import SimpleNamespace

gen_config = SimpleNamespace(
    n_features=10,
    n_causal_range=[3, 5],
    rule_families=["linear", "threshold", "tree"],
    spurious_strength_range=[0.80, 0.90],
    label_noise=0.02,
    coefficient_scale=1.5,
)

N_TASKS = 5
N_POOL = 256
N_TEST_ID = 100
N_TEST_OOD = 100

SHIFT_TYPES = {
    "synth_covariate": "covariate",
    "synth_concept": "mechanism",
    "synth_spurious": "spurious_reversal",
}

tasks = _sample_tasks(gen_config, N_TASKS, "synth", gen_config.rule_families) # return the "framework/blueprint" for a set of tasks
print(f"Generated {len(tasks)} tasks:")
for t in tasks:
    print(f"  {t.task_id}: family={t.rule_family}, "
          f"causal={t.causal_features}, spur_str={t.spurious_strength:.2f}")

## 2. Build datasets

For each shift type, we aggregate all tasks into one dataset.
The train pool and test sets are concatenated across tasks, with a `task_id` column
to allow per-task analysis.


In [ ]:
datasets = {}

for ds_name, env_ood in SHIFT_TYPES.items():
    all_pool, all_test_id, all_test_ood = [], [], []

    for i, task in enumerate(tasks):
        pool, tid, tood = task_to_dataframes(
            task, env_ood,
            n_pool=N_POOL, n_test_id=N_TEST_ID, n_test_ood=N_TEST_OOD,
            seed=SEED + i,
        )
        pool["task_id"] = task.task_id
        tid["task_id"] = task.task_id
        tood["task_id"] = task.task_id
        all_pool.append(pool)
        all_test_id.append(tid)
        all_test_ood.append(tood)

    pool_df = pd.concat(all_pool, ignore_index=True)
    test_id_df = pd.concat(all_test_id, ignore_index=True)
    test_ood_df = pd.concat(all_test_ood, ignore_index=True)

    datasets[ds_name] = {"pool": pool_df, "test_id": test_id_df, "test_ood": test_ood_df}

    print(f"\n{ds_name} ({env_ood}):")
    print(f"  pool:     {len(pool_df):>5d} rows, label_1_rate={pool_df['label'].mean():.3f}")
    print(f"  test_id:  {len(test_id_df):>5d} rows, label_1_rate={test_id_df['label'].mean():.3f}")
    print(f"  test_ood: {len(test_ood_df):>5d} rows, label_1_rate={test_ood_df['label'].mean():.3f}")

In [ ]:
datasets

## 3. Visualise feature distributions (ID vs OOD)


In [ ]:
fig, axes = plt.subplots(len(SHIFT_TYPES), 4, figsize=(16, 3.5 * len(SHIFT_TYPES)))

for row, (ds_name, data) in enumerate(datasets.items()):
    show_feats = ["f0", "f1", "f2", "f8"]
    for col, feat in enumerate(show_feats):
        ax = axes[row, col]
        ax.hist(data["test_id"][feat], bins=30, alpha=0.5, density=True, label="ID", color="steelblue")
        ax.hist(data["test_ood"][feat], bins=30, alpha=0.5, density=True, label="OOD", color="coral")
        ax.set_title(f"{ds_name} — {feat}" if col == 0 else feat)
        ax.legend(fontsize=8)
        if col == 0:
            ax.set_ylabel(ds_name, fontsize=10, fontweight="bold")

fig.suptitle("Feature distributions: ID vs OOD", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Label distribution comparison


In [ ]:
label_stats = []
for ds_name, data in datasets.items():
    for env, key in [("pool", "pool"), ("ID", "test_id"), ("OOD", "test_ood")]:
        df = data[key]
        label_stats.append({
            "dataset": ds_name, "split": env,
            "n": len(df),
            "label_1_rate": df["label"].mean(),
            "label_0_rate": 1 - df["label"].mean(),
        })

label_df = pd.DataFrame(label_stats)

fig, axes = plt.subplots(1, len(SHIFT_TYPES), figsize=(5 * len(SHIFT_TYPES), 4))
for i, ds_name in enumerate(SHIFT_TYPES):
    sub = label_df[label_df["dataset"] == ds_name]
    axes[i].bar(sub["split"], sub["label_1_rate"], color=["steelblue", "steelblue", "coral"])
    axes[i].set_title(ds_name)
    axes[i].set_ylabel("P(Y=1)")
    axes[i].set_ylim(0, 1)
    axes[i].axhline(0.5, ls="--", color="grey", alpha=0.5)

fig.suptitle("Label distribution by split", fontsize=14)
plt.tight_layout()
plt.show()

print(label_df.to_string(index=False))

## 5. Shift magnitude diagnostics

Quantify how much each shift type moves the joint distribution P(X, Y),
using a domain discriminator trained on **features + label + feature×label interactions**.
Concept shift flips the direction of feature-label correlations while keeping marginals
identical (X is symmetric N(0,1)), so explicit interaction terms are needed to detect it.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

def build_discriminator_features(df):
    """Features + label + feature*label interactions for domain discrimination."""
    feats = df[FEATURE_NAMES].values
    label = df["label"].values.reshape(-1, 1)
    interactions = feats * label
    return np.hstack([feats, label, interactions])

print("Domain discriminator PR-AUC (higher = more detectable shift):")
print("Features + label + feature×label interactions")
print("=" * 70)

for ds_name, data in datasets.items():
    X_id = build_discriminator_features(data["test_id"])
    X_ood = build_discriminator_features(data["test_ood"])
    X = np.vstack([X_id, X_ood])
    y_domain = np.array([0] * len(X_id) + [1] * len(X_ood))

    clf = HistGradientBoostingClassifier(max_iter=100, max_depth=4, random_state=SEED)
    scores = cross_val_score(clf, X, y_domain, cv=5, scoring="average_precision")
    print(f"  {ds_name:25s}  PR-AUC = {scores.mean():.3f} +/- {scores.std():.3f}")

## 6. Save datasets to disk

Each dataset is saved in the same format as the real-arm data:
`train_pool.parquet`, `test_id.parquet`, `test_ood.parquet`, plus JSON metadata.


In [ ]:
for ds_name, data in datasets.items():
    out_dir = DATA_ROOT / ds_name
    save_dataset(data["pool"], data["test_id"], data["test_ood"], out_dir)

    task_meta = []
    for t in tasks:
        task_meta.append({
            "task_id": t.task_id, "rule_family": t.rule_family,
            "causal_features": t.causal_features,
            "spurious_strength": t.spurious_strength,
        })
    with open(out_dir / "task_meta.json", "w") as f:
        json.dump(task_meta, f, indent=2, default=str)

    print(f"Saved {ds_name} -> {out_dir}")

print(f"\nAll datasets saved under {DATA_ROOT}")

## 7. Serialisation preview

Check what the LLM will actually see for a synthetic row.


In [ ]:
from src.data.serialisation import serialise_row, ordered_feature_names

sample_row = datasets["synth_covariate"]["pool"].iloc[0]
codebook = make_codebook()
feat_dict = {f: sample_row[f] for f in FEATURE_NAMES}
ordered = ordered_feature_names(feat_dict)

demo_text = serialise_row({f: sample_row[f] for f in ordered}, label=str(int(sample_row["label"])), codebook=codebook)
query_text = serialise_row({f: sample_row[f] for f in ordered}, codebook=codebook)

print("Demo (with label):")
print(demo_text)
print("\nQuery (no label):")
print(query_text)

In [ ]:
from src.inference.prompts import build_chat_messages

task_desc, _, lm0, lm1 = SYNTHETIC_TASK_DESC
messages = build_chat_messages(
    task_desc, LABEL_TOKENS,
    demo_lines=[demo_text],
    query_line=query_text,
    label_meanings=(lm0, lm1),
)

for msg in messages:
    print(f"--- {msg['role'].upper()} ---")
    print(msg['content'][:500])
    print()

# Trying with different formats for the query


Similar to how Min et al have shown, as an extension, we could see if this has an effects as they have shown and if it increases performance ?
